## Fine tuning BERT for LLM hallucination

# Experimentation 

## 1. Getting data

In [1]:
from datasets import load_dataset

# Load a specific language (e.g., English)
dataset = load_dataset("Helsinki-NLP/mu-shroom", "all")

# Access splits
train = dataset["train_unlabeled"]
val = dataset["validation"]
test = dataset["test"]


In [2]:
train.__len__

<bound method Dataset.__len__ of Dataset({
    features: ['id', 'lang', 'model_input', 'model_output_text', 'model_id', 'wikipedia_url', 'soft_labels', 'hard_labels', 'model_output_logits', 'model_output_tokens', 'annotations'],
    num_rows: 3351
})>

In [3]:
# Get an example from validation set
example = dataset["validation"][100]

print(f"Language: {example['lang']}")
print(f"Input: {example['model_input']}")
print(f"Model Output: {example['model_output_text']}")
print(f"Hallucination spans: {example['hard_labels']}")

# Visualize hallucination spans
text = example["model_output_text"]
for span in example["hard_labels"]:
    start, end = span
    print(f"Hallucinated text: '{text[start:end]}'")


Language: EN
Input: What did Petra van Staveren win a gold medal for?
Model Output: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
Hallucination spans: [[25, 31], [45, 49], [69, 83]]
Hallucinated text: 'silver'
Hallucinated text: '2008'
Hallucinated text: 'Beijing, China'


## 2. Tokenization

Examplified

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-uncased")

print(f"Original: {text}")
print(f"Tokenized: {tokenizer.tokenize(text)}")
print(f"Token ID: {tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))}")

Original: Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.
Tokenized: ['petra', 'van', 'sto', '##veren', 'won', 'a', 'silver', 'medal', 'in', 'the', '2008', 'summer', 'olympics', 'in', 'beijing', ',', 'china', '.']
Token ID: [26135, 10147, 12291, 67425, 11441, 143, 15143, 13543, 10104, 10103, 10224, 13138, 19450, 10104, 20885, 117, 11707, 119]


##### Illustration
`offset_mapping`
- So while tokenizing we are getting the idex for starting and ending of each and every word in our given text
- ex: Petra van Stoveren -> [(0,5), (6,9), (10,15)]

In [5]:
text = example["model_output_text"]
char_spans = example["hard_labels"]  # list of [start,end] spans
enc = tokenizer(text, return_offsets_mapping=True)
offsets = enc.offset_mapping
# Initialize label for every word as -100
labels = [-100] * len(offsets)
for (span_start, span_end) in char_spans:
    for i,(s,e) in enumerate(offsets):
        if e <= span_start or s >= span_end:
            continue
        # label first sub-token of each word overlapping a span
        if s == 0 and e != 0:
            labels[i] = 1


In [6]:
train[0]

{'id': None,
 'lang': 'EN',
 'model_input': 'Do all arthropods have antennae?',
 'model_output_text': ' Yes, all insects and arachnids (including spiders, scorpions, and ticks) have at least one antenna. ',
 'model_id': 'togethercomputer/Pythia-Chat-Base-7B',
 'wikipedia_url': None,
 'soft_labels': None,
 'hard_labels': None,
 'model_output_logits': [-2.57427001,
  5.1865358353,
  5.4173498154,
  2.3298389912,
  6.6413722038,
  3.5858919621,
  10.0333786011,
  4.0347309113,
  2.5324678421,
  0.595354557,
  2.0120403767,
  1.8963916302,
  6.8019108772,
  6.2508711815,
  4.6549735069,
  2.6137938499,
  -3.9706561565,
  7.5047488213,
  4.2792539597,
  3.0500841141,
  -0.4772415757,
  3.1624026299,
  13.7479448318,
  -1.7824890614,
  3.9432423115,
  4.9513559341,
  4.3375749588,
  14.8357839584],
 'model_output_tokens': ['ĠYes',
  ',',
  'Ġall',
  'Ġinsects',
  'Ġand',
  'Ġar',
  'ach',
  'n',
  'ids',
  'Ġ(',
  'including',
  'Ġspiders',
  ',',
  'Ġsc',
  'orp',
  'ions',
  ',',
  'Ġand',

In [7]:
# To install: pip install tavily-python
from tavily import TavilyClient
client = TavilyClient("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")
response = client.search(
    query="Do all arthropods have antennae?",
    include_answer="basic",
    search_depth="basic"
)
answer = response.get("answer", "No answer returned")
print(answer)

Not all arthropods have antennae; crustaceans have two pairs, while most others have one pair, except chelicerates and proturans which lack them entirely.


In [8]:
response["answer"]

'Not all arthropods have antennae; crustaceans have two pairs, while most others have one pair, except chelicerates and proturans which lack them entirely.'

In [ ]:
####################  DONT RUN THIS IT RESET JSON
# import json
# reference_map = {
#     "Do all arthropods have antennae?": "No, not all arthropods have antennae. Insects and crustaceans usually have antennae, but some arthropods such as spiders and scorpions do not.",
#     "What is photosynthesis?": "Photosynthesis is the process by which plants convert light energy into chemical energy using carbon dioxide and water, producing glucose and oxygen.",
#     "What is the capital of France?": "The capital of France is Paris.",
#     "How does machine learning work?": "Machine learning uses algorithms to learn patterns from data and make predictions or decisions without being explicitly programmed for every case."
# }

# # Save as JSON
# with open("reference_map.json", "w", encoding="utf-8") as f:
#     json.dump(reference_map, f, indent=4, ensure_ascii=False)

In [12]:
import json

with open("reference_map.json", "r", encoding="utf-8") as f:
    reference_map = json.load(f)

answer = reference_map["Do all arthropods have antennae?"]
print(answer)

No, not all arthropods have antennae. Insects and crustaceans usually have antennae, but some arthropods such as spiders and scorpions do not.


# Implementation

In [13]:
from dotenv import load_dotenv
import os
import json

# Load .env file
load_dotenv()

# Read variables
API_Tavily = os.getenv("API_Tavily")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


### Step 1: Get the real answers from Tavily

In [14]:
# The data had many duplicate answers so we need to get only unique Q to save api cost
unique_questions = list(set(train["model_input"]))
print(len(unique_questions))
print(len(train))

201
3351


In [15]:
# translate if answers are in different language
import os
from langdetect import detect, DetectorFactory
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

DetectorFactory.seed = 0


# Groq LLM
model = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    # other params...
)


def detect_language(text):
    """
    Detect language using ISO-639-1 codes:
    en, de, fr, hi, etc.
    """
    try:
        return detect(text)
    except Exception:
        return "unknown"


def translate_text(text, target_language):
    messages = [
    ("system", f"You are a helpful translator. Translate the user sentence to {target_language}."),
    ("human", text),
    ]
    response = model.invoke(messages)
    return response.content.strip()


def normalize_answer(question, answer):
    """
    Check if question and answer languages match.
    Translate answer if needed.
    """

    question_lang = detect_language(question)
    answer_lang = detect_language(answer)

    if question_lang == answer_lang:
        return answer

    print("Language mismatch → translating")

    return translate_text(
        answer,
        question_lang
    )

In [16]:
# search for true answers

from tavily import TavilyClient

client = TavilyClient(API_Tavily)

# Get reference first old else new
if os.path.exists("reference_map.json"):
    with open("reference_map.json", "r", encoding="utf-8") as f:
        reference_map = json.load(f)
else:
    reference_map = {}

# Search questions
for question in unique_questions:
    # Skip already searched questions
    if question in reference_map:
        print(f"Skipping: {question}")
        continue

    # Get answers
    try:
        response = client.search(
            query=question,
            include_answer="basic",
            search_depth="basic"
        )
        answer = response.get("answer") or "No answer returned"
        answer = normalize_answer(question,answer)
        reference_map[question] = answer
        
        # Save immediately after each successful search
        with open("reference_map.json", "w", encoding="utf-8") as f:
            json.dump(reference_map, f, indent=4, ensure_ascii=False)
        print(f"Saved: {question}")
        
    except Exception as e:
        print(f"Failed for {question}: {e}")
        continue

Saved: In which country is the Salzburg Red Bull Arena?
Saved: Il y a-t-il une importante présence sikhe à Ludhiana?
Saved: Quel rôle ont les tours de silence dans le zoroastrisme?
Saved: Pour quelles raisons le kremlin de Tobolsk est-il notoire?
Saved: In which year were the Gampel and Bratsch municipalities merged?
Saved: Quel est le lien entre la ville de Nahavand et Laodicée?
Language mismatch → translating
Saved: 摩鹿加云斑蛛分布于哪些国家和地区？
Saved: When was Toruń part of the Hanseatic League?
Language mismatch → translating
Saved: ¿Cuál es el nombre real del rapero conocido como Chance the Rapper?
Language mismatch → translating
Saved: Combien pèse l’hélicoptère Ingenuity?
Saved: ¿Cuántos premios ha ganado Dave Franco por actuaciones en su carrera como actor?
Saved: When was Bischofsheim a constitutent community of Mainz?
Saved: 波兰托伦市有什么受欢迎的旅游元素？
Saved: ¿En qué siglos vivió el conde sueco Carl Löwenheim?
Language mismatch → translating
Saved: Petra van Staveren a-t-elle remporté une médaille

In [25]:
def add_reference(example):
    example["reference_answer"] = reference_map[example["model_input"]]
    return example

train = train.map(add_reference)

# now check a sample AFTER mapping
print(train[1])

Map:   0%|          | 0/3351 [00:00<?, ? examples/s]

{'id': None, 'lang': 'EN', 'model_input': 'Do all arthropods have antennae?', 'model_output_text': ' Yes, all insects and arachnids have at least one antenna. ', 'model_id': 'togethercomputer/Pythia-Chat-Base-7B', 'wikipedia_url': None, 'soft_labels': None, 'hard_labels': None, 'model_output_logits': [-2.57427001, 5.1865358353, 5.4173498154, 2.3298389912, 6.6413722038, 3.5858919621, 10.0333786011, 4.0347309113, 2.5324678421, 0.595354557, 3.7296099663, 14.31341362, -1.5164659023, 3.9656012058, 4.8837690353, 3.5321044922, 15.5193033218], 'model_output_tokens': ['ĠYes', ',', 'Ġall', 'Ġinsects', 'Ġand', 'Ġar', 'ach', 'n', 'ids', 'Ġhave', 'Ġat', 'Ġleast', 'Ġone', 'Ġantenna', '.', 'Ċ'], 'annotations': [{'annotator_id': '', 'labels': []}], 'reference_answer': 'No, not all arthropods have antennae. Insects and crustaceans usually have antennae, but some arthropods such as spiders and scorpions do not.'}


- Now we have answers which are reliable and we are going to use an element in order to generate the labels. 

### Step two: using an LLM to generate the soft and hard labels 

In [137]:
import random

filtered = train.filter(lambda x: x["lang"] in ["EN", "HI"])

idx = random.randint(0, len(filtered) - 1)
sample = filtered[idx]

print("lang:", sample["lang"])
print("model_input:", sample["model_input"])
print("model_output_text:", sample["model_output_text"])

lang: EN
model_input: Who was the father of Carl Löwenhielm?
model_output_text: Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917). Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.


### Prompts

In [138]:
# default
message1 = [
(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- hallucinated_span must be copied EXACTLY from the LLM answer.
- Do not paraphrase or correct the text.
- Include only false or unsupported factual claims.
- Include complete factual claims that are false or unsupported.
- The spans must be as short as possible, but include the complete factual claim that is false.
- If no hallucination exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]
# Atomic claim decomposition
message2 = [
(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- Copy spans exactly from the LLM answer.
- Identify each individual false factual claim separately.
- If a sentence contains multiple hallucinated facts, return separate spans.
- Do not include correct parts of the sentence.
- Do not mark equivalent statements as hallucinations.
- If no hallucination exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]
# Strict contradiction detector
message3 = [
(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- Copy spans exactly from the LLM answer.
- Only return claims that directly contradict the reference answer.
- Do not mark missing information as hallucination.
- Do not mark different wording with the same meaning as hallucination.
- Avoid guessing.
- If no direct contradiction exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]
# Complete factual claim
message4 = [
(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- Copy spans exactly from the LLM answer.
- Return complete factual claims that are false or unsupported.
- Include enough context to understand why the claim is wrong.
- Do not return isolated words or numbers.
- Do not mark correct paraphrases as hallucinations.
- If no hallucination exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]
# smallest hall
message5 = [
(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- Copy spans exactly from the LLM answer.
- Return only statements that are factually incorrect or unsupported.
- Select the smallest span that contains the hallucinated information.
- Do not include surrounding correct information.
- Do not mark paraphrases or equivalent meanings as hallucinations.
- If no hallucination exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]
# Context-aware balanced annotation
message6 = [(
"system",
"You are a hallucination detection assistant. Identify factual claims in an LLM-generated answer that are hallucinated when compared with the reference answer."
),
(
"user",
f"""
Compare the LLM answer with the reference answer.

Return ONLY a Python list of tuples:
[hallucinated_span]

Rules:
- Copy spans exactly from the LLM answer.
- Return the shortest span that still represents a complete false factual claim.
- Consider the meaning of the statement, not only exact wording.
- Do not mark correct explanations, paraphrases, or logical consequences of the reference answer.
- Do not include unsupported speculation unless it conflicts with the reference answer.
- If no hallucination exists, return [].
- Do not provide explanations.

Question:
{sample['model_input']}

LLM Answer:
{sample['model_output_text']}

Reference Answer:
{sample['reference_answer']}
"""
)
]

messages = [message1,message2,message3,message4,message5,message6]

## Qwen model 

In [139]:
llm = ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0,
    max_tokens=None,
    reasoning_effort="none",
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    
)
for i,message in enumerate(messages):
    ai_msg = llm.invoke(message)
    print(f"Message {i}:-\n")
    print(f"Question- {sample['model_input']}\n "
    f"LLM Ans-  {sample['model_output_text']}\n"
    f"True Ans- {sample['reference_answer']}\n"
    f"{ai_msg.content}")

Message 0:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917). Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.
True Ans- Carl Löwenhielm's father was Fredrik Adolf Löwenhielm. His mother was Augusta von Fersen. He was an illegitimate son of King Charles XIII of Sweden.
[("His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917).", "Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.", "He was born on March 1, 1866, in Stockholm, Sweden.", "Carl Löwenhielm was a Swedish military officer and politician.")]
Message 1:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockh

## gpt-oss-20b

In [140]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=None,
    reasoning_effort="low",
    reasoning_format="hidden",
    timeout=None,
    max_retries=2,
)
for i,message in enumerate(messages):
    ai_msg = llm.invoke(message)
    print(f"Message {i}:-\n")
    print(f"Question- {sample['model_input']}\n "
    f"LLM Ans-  {sample['model_output_text']}\n"
    f"True Ans- {sample['reference_answer']}\n"
    f"{ai_msg.content}")

Message 0:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917). Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.
True Ans- Carl Löwenhielm's father was Fredrik Adolf Löwenhielm. His mother was Augusta von Fersen. He was an illegitimate son of King Charles XIII of Sweden.
[("He was born on March 1, 1866, in Stockholm, Sweden."), ("His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917)."), ("Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.")]
Message 1:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-19

## Llama-3.3-70b-versatile

In [141]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
for i,message in enumerate(messages):
    ai_msg = llm.invoke(message)
    print(f"Message {i}:-\n")
    print(f"Question- {sample['model_input']}\n "
    f"LLM Ans-  {sample['model_output_text']}\n"
    f"True Ans- {sample['reference_answer']}\n"
    f"{ai_msg.content}")

Message 0:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917). Therefore, Johan Henrik Löwenhielm was the father of Carl Löwenhielm.
True Ans- Carl Löwenhielm's father was Fredrik Adolf Löwenhielm. His mother was Augusta von Fersen. He was an illegitimate son of King Charles XIII of Sweden.
[("Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917)"), ("Johan Henrik Löwenhielm was the father of Carl Löwenhielm")]
Message 1:-

Question- Who was the father of Carl Löwenhielm?
 LLM Ans-  Carl Löwenhielm was a Swedish military officer and politician. He was born on March 1, 1866, in Stockholm, Sweden. His parents were Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917). Therefore, Johan Henrik Löwenhie

### Word spans to indexes

In [143]:
import ast

output = ast.literal_eval(ai_msg.content)

text = sample["model_output_text"]

results = []

for span in output:
    start = text.find(span)

    if start == -1:
        print("Not found:", span)
        continue

    end = start + len(span) - 1

    results.append({
        "span": span,
        "char_start": start,
        "char_end": end,
        "label": 1
    })

print(results)

[{'span': 'Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917)', 'char_start': 132, 'char_end': 216, 'label': 1}, {'span': 'Johan Henrik Löwenhielm was the father of Carl Löwenhielm', 'char_start': 230, 'char_end': 286, 'label': 1}]


In [146]:
s =  sample["model_output_text"]
s[132:217]

'Johan Henrik Löwenhielm (1832-1905) and Hedvig Charlotta Amalia von Essen (1840-1917)'

"[('from 1972 until 1980', 1.0), ('The merger took place on January 1, 1972', 1.0), ('Bischofsheim became a part of Mainz-Bischofsheim, which was then a separate city until it was reincorporated into Mainz on November 7, 1980', 1.0)]"

In [ ]:
from groq import Groq

client = Groq()
completion = client.chat.completions.create(
    model="qwen/qwen3.6-27b",
    messages=[
      {
        "role": "user",
        "content": "Can you just output the words which are hallucinated by LLM?(i dont want reasoning)\n\nQuestion- What did Petra van Staveren win a gold medal for? LLM Ans- Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China. True Ans- Petra van Staveren won the gold medal in the 100 meter breaststroke at the 1984 Summer Olympics in Los Angeles"
      },
      {
        "role": "assistant",
        "content": "\n<think>\nHere's a thinking process:\n\n1.  **Analyze User Input:**\n   - **Question:** What did Petra van Staveren win a gold medal for?\n   - **LLM Answer:** Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.\n   - **True Answer:** Petra van Staveren won the gold medal in the 100 meter breaststroke at the 1984 Summer Olympics in Los Angeles\n   - **User Request:** \"Can you just output the words which are hallucinated by LLM?(i dont want reasoning)\"\n\n2.  **Identify Hallucinations:**\n   - Compare LLM answer with True answer.\n   - LLM says: \"silver medal\", \"2008 Summer Olympics\", \"Beijing, China\"\n   - True says: \"gold medal\", \"100 meter breaststroke\", \"1984 Summer Olympics\", \"Los Angeles\"\n   - Also, LLM misspells the name: \"Stoveren\" vs \"Staveren\" (though the prompt says \"Petra van Staveren\", LLM says \"Petra van Stoveren\")\n   - The hallucinated parts are the incorrect facts: \"silver medal\", \"2008\", \"Beijing, China\", and the misspelled name \"Stoveren\".\n   - The user wants *just* the hallucinated words. I need to extract the specific incorrect tokens/phrases from the LLM answer.\n\n   Let's list them clearly:\n   - silver\n   - 2008\n   - Beijing\n   - China\n   - Stoveren (misspelling)\n\n   Wait, the prompt says \"output the words which are hallucinated\". I should just list the incorrect words/phrases directly from the LLM's answer.\n   LLM answer: \"Petra van Stoveren won a silver medal in the 2008 Summer Olympics in Beijing, China.\"\n   Hallucinated parts: \"Stoveren\", \"silver\", \"2008\", \"Beijing\", \"China\"\n   Also, the event \"100 meter breaststroke\" is missing, but that's an omission, not a hallucination. Hallucination refers to fabricated/incorrect information.\n\n   Let's verify facts: Petra van Staveren is a Dutch swimmer. She won gold in 100m breaststroke at 1984 Olympics. The LLM completely made up silver, 2008, Beijing, China, and misspelled her name.\n\n   The user wants just the words. I will output them as a comma-separated list or just the exact words.\n   \"Stoveren, silver, 2008, Beijing, China\"\n\n   Should I include \"won a ... medal in the ... Summer Olympics in ...\"? No, just the hallucinated content.\n   I'll output: Stoveren, silver, 2008, Beijing, China\n\n   Double-check: The prompt says \"output the words which are hallucinated by LLM?(i dont want reasoning)\". I will strictly follow that.\n   I'll just list them plainly.\n   \"Stoveren, silver, 2008, Beijing, China\"\n\n   Wait, is \"Summer Olympics\" hallucinated? No, it's a real event type, just wrong year/location.\n   I'll stick to the clearly wrong facts.\n   Output: Stoveren, silver, 2008, Beijing, China\n   That's it. No extra text.\n   Let's verify format: \"Stoveren, silver, 2008, Beijing, China\" matches the request.\n   I will output exactly that.✅\n   One minor thing: The LLM said \"Stoveren\" instead of \"Staveren\". That's a hallucination/typo. I'll include it.\n   Final: Stoveren, silver, 2008, Beijing, China\n   Ready.✅\n   Output matches request. No reasoning. Just the words.✅\n   Proceed. \n   Output: Stoveren, silver, 2008, Beijing, China\n   (Self-Correction/Refinement during thought)\n   Should I include the exact phrase from the LLM? The prompt says \"words which are hallucinated\". I'll just list them.\n   \"Stoveren, silver, 2008, Beijing, China\" is precise.\n   I'll output exactly that.✅\n   Done. \n   Output Generation. \n   [Output]: Stoveren, silver, 2008, Beijing, China\n   (Note: I'll make sure it's just the words as requested.)✅\n   All good. \n   Proceeds. \n   [Final Check of the Prompt]: \"Can you just output the words which are hallucinated by LLM?(i dont want reasoning)\"\n   Matches.✅\n   Output: Stoveren, silver, 2008, Beijing, China\n   (I could also format it as a simple list, but comma-separated is fine.)\n   Done. \n   [Output Generation] -> Stoveren, silver, 2008, Beijing, China\n   *(Self-Correction/Verification during generation prep)*: \n   Wait, is \"Stoveren\" considered a hallucination or just a typo? In LLM evaluation, misspelling a proper noun in a factual context is often counted as a hallucination/error. I'll include it. \n   The core hallucinated facts are: silver, 2008, Beijing, China. \n   I'll output: Stoveren, silver, 2008, Beijing, China\n   Perfect.✅\n   Output matches. \n   [Proceeds]\n</think>\n\nStoveren, silver, 2008, Beijing, China"
      },
      {
        "role": "user",
        "content": ""
      }
    ],
    temperature=0.0,
    max_completion_tokens=2048,
    reasoning_effort="default",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")
